# 09 — COVID Resilience Analysis

This notebook analyzes how organic waste ban states fared during the
COVID-19 pandemic compared to non-ban states. We examine:
- The 2019-2020 drop in landfill tonnage
- The 2020-2022 recovery trajectory

Output: `covid_resilience.png`

---
## 1. Setup & Build Panel

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import pandas as pd, numpy as np, warnings
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from utils.causal_inference import build_state_year_panel

BAN_COLOR, NOBAN_COLOR = '#2C5F2D', '#888888'

os.makedirs('outputs/causal/charts', exist_ok=True)

In [3]:
state_year, policy = build_state_year_panel()
print(f"Panel shape: {state_year.shape}")
print(f"Policy states: {len(policy)}")
state_year.head(3)

Panel shape: (750, 18)
Policy states: 12


,state,year,tons_surplus,tons_waste,tons_landfill,tons_composting,tons_donations,tons_anaerobic_digestion,tons_animal_feed,us_dollars_surplus,co2e,water,meals_wasted,landfill_rate,diversion_rate,ban_year,ban_active,is_ban_state
0,Alabama,2010,605567.602551,526539.336201,307629.284782,101596.317651,14262.997676,2945.070690,40430.831922,3.317411e+09,2.722035e+06,1.992007e+11,9.855077e+08,0.584247,0.302418,9999,0,False
1,Alabama,2011,625989.926059,542326.715942,316125.547806,105286.414382,15274.325194,3160.703920,42921.350342,3.432795e+09,2.815341e+06,2.058467e+11,1.017859e+09,0.582906,0.307274,9999,0,False
2,Alabama,2012,643898.998395,557263.486746,325597.048581,107782.742068,16205.356765,3334.223213,43874.216529,3.566739e+09,2.903121e+06,2.134882e+11,1.046156e+09,0.584278,0.307209,9999,0,False


---
## 2. COVID Impact Analysis

We isolate 2019-2022 to measure the pandemic shock (2019-2020 change)
and subsequent recovery (2020-2022 change), comparing ban vs. non-ban states.

In [4]:
covid = state_year[state_year['year'].isin([2019, 2020, 2021, 2022])]
piv = covid.pivot_table(values='tons_landfill', index='state', columns='year')
piv['drop_2020'] = (piv[2020] - piv[2019]) / piv[2019] * 100
piv['recovery_2022'] = (piv[2022] - piv[2020]) / piv[2020] * 100
piv['is_ban'] = piv.index.isin(policy['state'].values)

print(f"Ban states — Avg 2020 drop: {piv[piv['is_ban']]['drop_2020'].mean():.1f}%")
print(f"Non-ban    — Avg 2020 drop: {piv[~piv['is_ban']]['drop_2020'].mean():.1f}%")
print(f"Ban states — Avg 2022 recovery: {piv[piv['is_ban']]['recovery_2022'].mean():.1f}%")
print(f"Non-ban    — Avg 2022 recovery: {piv[~piv['is_ban']]['recovery_2022'].mean():.1f}%")

Ban states — Avg 2020 drop: -27.9%
Non-ban    — Avg 2020 drop: -40.4%
Ban states — Avg 2022 recovery: 70.7%
Non-ban    — Avg 2022 recovery: 55.9%


---
## 3. Resilience Visualization

Side-by-side boxplots comparing the distribution of COVID impact
and recovery rates between ban and non-ban states.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for i, (col, title) in enumerate([
    ('drop_2020', '2019-2020 Change (%)'),
    ('recovery_2022', '2020-2022 Recovery (%)')
]):
    bp = axes[i].boxplot(
        [piv[piv['is_ban']][col], piv[~piv['is_ban']][col]],
        labels=['Ban', 'Non-Ban'], patch_artist=True, widths=0.6
    )
    bp['boxes'][0].set_facecolor(BAN_COLOR)
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(NOBAN_COLOR)
    bp['boxes'][1].set_alpha(0.7)
    axes[i].set_title(title)
    axes[i].axhline(y=0, color='black', lw=0.5)
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle('COVID Resilience: Ban vs Non-Ban States', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/causal/charts/covid_resilience.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved covid_resilience.png')

Saved covid_resilience.png
